# What

There is data used by the pytests to test the functionality of the engine and viewer.

I have previously somewhat randomly and manually created these data subsets to be used.

It would be better to have it in one place so that it is clear what data is used and needed.

## Modules

In [ ]:
import pandas as pd 
import os
import shutil
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()

This is going to be ran assuming there is a output folder that has the expected files.

Rather than hook this up to the config file it is just a quick and simple hack to make it more transparent.

In [ ]:
root_folder = Path('../../')
output_folder = root_folder / 'output'
test_data_folder = root_folder / 'tests' / 'data'
test_output_folder = test_data_folder / 'output'

# Website report tables

When web scraping TAIC uses a pickle file to store the report table it has gathered so far.

ATSB uses a simliar one.



In [ ]:
taic_reports_table = pd.read_pickle(os.path.join(output_folder, 'taic_website_reports_table.pkl'))
taic_reports_table.iloc[25:].to_pickle(os.path.join(test_output_folder, 'taic_website_reports_table.pkl'))

atsb_reports_table = pd.read_pickle(os.path.join(output_folder, 'atsb_website_reports_table.pkl'))
atsb_reports_table.sort_values(by='year', ascending=False).iloc[25:].to_pickle(os.path.join(test_output_folder, 'atsb_website_reports_table.pkl'))

## Report titles

In [ ]:
titles = pd.read_pickle(os.path.join(test_output_folder, "report_titles.pkl"))

# Getting report PDFs

This is used by the `test_PDFParser.py`

In [ ]:
# Create the test pdfs. These will be stored in the 'test-stable

import tempfile
from engine.utils.AzureStorage import PDFStorageManager


test_report_ids_for_pdf = set([
    "ATSB_r_2021_010",
    "ATSB_r_2021_004",    
    "ATSB_a_2007_030",
    "ATSB_r_2010_007",
    "ATSB_a_2002_646",
    "TSB_a_2022_O0118",
    "TSB_m_2021_A0041",
    "TSB_a_2011_F0012",
    "TAIC_r_2014_103",
    "TAIC_r_2004_121",
    "TAIC_a_2019_006",
])

stable_manager = PDFStorageManager(
    os.environ["AZURE_STORAGE_ACCOUNT_NAME"],
    os.environ["AZURE_STORAGE_ACCOUNT_KEY"],
    "test-stable-reportpdfs"
)

prod_manager = PDFStorageManager(
    os.environ["AZURE_STORAGE_ACCOUNT_NAME"],
    os.environ["AZURE_STORAGE_ACCOUNT_KEY"],
    "prod-reportpdfs"
)

# Find out which ones are not already in the stable container.
current_pdfs_in_stable = set(stable_manager.list_pdfs())

# Find out which ones should be 
pdfs_to_upload = test_report_ids_for_pdf - current_pdfs_in_stable
pdfs_to_delete = current_pdfs_in_stable - test_report_ids_for_pdf

# Delete any that are in the stable container but shouldn't be.
for pdf_file in pdfs_to_delete:
    stable_manager.delete_pdf(pdf_file)
    print(f"Deleted {pdf_file}")


# For ones to upload download to temp file and upload to stable container.
for report_to_upload in pdfs_to_upload:
    pdf_bytes = prod_manager.download_pdf(report_to_upload)
    stable_manager.upload_pdf(report_to_upload, pdf_bytes)
    print(f"Uploaded {report_to_upload}")

## Creating the extracted reports data

This is used by:
- `test_RecommendationSafetyIssueLinking.py`
- `test_RecommendationResponseClassification.py`
- `test_Embedding.py`


In [ ]:
extracted_reports = pd.read_pickle(os.path.join(output_folder, "extracted_reports.pkl"))

extracted_reports.sample(n=50, random_state=42).to_pickle("../../tests/data/output/extracted_reports.pkl")

# Creating report text dataset

This is used by the `test_ReportExtracting.py`

In [ ]:
report_text = pd.read_pickle(os.path.join(output_folder, "parsed_reports.pkl"))

report_text.set_index("report_id",inplace=True)

report_text

In [ ]:
ids = [
    "TAIC_m_2016_204",
    "TAIC_m_2020_202",
    "TAIC_r_2014_102",
    "TAIC_a_2014_004",
    "TAIC_m_2010_204",
    "TAIC_a_2010_001",
    "TAIC_r_2022_101",
    "TAIC_a_2010_009",
    "TAIC_r_2019_106",
]

# This is added as this is what was used in the previous extracted set which is used by alot of tests.
ids.extend([
 'TAIC_m_2016_205',
 'TAIC_r_2002_122',
 'TAIC_r_2005_107',
 'TAIC_r_2004_113',
 'TAIC_a_2018_006',
 'TAIC_r_2001_104',
 'TAIC_r_2009_101',
 'TAIC_r_2012_102'])


ids.extend([
  "ATSB_m_2000_157",
  "ATSB_a_2023_011",
  "ATSB_a_2007_012",
  "ATSB_m_2001_170",
  "ATSB_r_2021_002"
 ])


ids.extend([
    "TSB_r_2020_V0230",
])

# Included to be used for content_page reading
ids.extend(
    [
        "TAIC_r_2019_102",
        "ATSB_a_2017_117", 
        "ATSB_a_2014_073",
        "TSB_m_2002_C0018",
        "TSB_a_2005_C0187",
        "ATSB_m_2017_003",
        "ATSB_a_2021_018",
        "TSB_a_2004_H0001",
        "TSB_a_2020_P0013"
    ]
)

# Included for testing recommendation extraction
ids.extend([
    "ATSB_a_2014_096",
    "ATSB_m_2013_011",
])


# Used for checking extraction of safety issues
ids.extend([
    "TAIC_m_2004_203",
    "TAIC_a_2020_003",

    "TSB_a_2023_W0096",

    # Old ATSB ones where website doesnt cover
    "ATSB_a_2005_912"
])

# Used for checking extraction of recommendations
ids.extend([
    "ATSB_a_2000_157",
    "ATSB_a_2020_007",
    "ATSB_r_2014_001",
    "ATSB_a_2007_018",
    "ATSB_m_2007_241",
    "ATSB_m_2022_007",
    "ATSB_r_2010_007",
    
])

filtered_report_text = report_text.loc[ids]


In [ ]:
filtered_report_text.to_pickle(os.path.join(test_output_folder, "parsed_reports.pkl"))

# Extracted reports

In [ ]:
extracted_reports = pd.read_pickle(os.path.join(output_folder, "extracted_reports.pkl"))

extracted_reports

In [ ]:
pd.read_pickle(os.path.join(test_output_folder, "extracted_reports.pkl"))

In [ ]:
extracted_reports.sample(n=50, random_state=42, ignore_index=True).to_pickle(os.path.join(test_output_folder, "extracted_reports.pkl"))

# Embeddings

In [ ]:
embedding_files = os.listdir(os.path.join(output_folder,"embeddings"))

embedding_dfs = [pd.read_pickle(os.path.join(output_folder, "embeddings", file)) for file in embedding_files]

embedding_dfs = [df.sample(n=10, random_state=42, ignore_index=True) for df in embedding_dfs]

os.makedirs(os.path.join(output_folder, "embeddings"), exist_ok=True)

for name, df in zip(embedding_files, embedding_dfs):
    df.to_pickle(os.path.join(test_output_folder, "embeddings", name))
    print(df)

## Vector db

In [ ]:
import dotenv
import engine.utils.EngineOutputStorage as EngineOutputStorage

dotenv.load_dotenv()

uploader = EngineOutputStorage.EngineOutputUploader(
    os.environ['AZURE_STORAGE_ACCOUNT_NAME'],
    os.environ['AZURE_STORAGE_ACCOUNT_KEY'],
    "engineoutput",
    None,
    "../../tests/data/vector_db",
    "../../output/embeddings/safety_issues_embeddings.pkl",
    "../../output/embeddings/recommendations_embeddings.pkl",
    "../../output/embeddings/report_sections_embeddings.pkl",
    "../../output/embeddings/report_text_embeddings.pkl",
)

uploader._upload_embeddings(sample_frac=0.01)

In [ ]:
import lancedb
vector_db = lancedb.connect("../../tests/data/vector_db")

table = vector_db.open_table("all_document_types")

data = table.to_pandas()

data

In [ ]:
data['document'].str.contains("work").sum()

## Response classification

In [ ]:
rec_classification = pd.read_pickle(os.path.join(output_folder, "recommendation_response_classification.pkl"))
rec_classification

In [ ]:
# ATSB website safety issues

atsb_safety_issues = pd.read_pickle(os.path.join(output_folder, "atsb_website_safety_issues.pkl"))

atsb_safety_issues[:-10].to_pickle(os.path.join(test_output_folder, "atsb_website_safety_issues.pkl"))

# Creat recommendation test data

In [ ]:
# Create recommendation test data

needed_ids = ['ATSB_a_2002_780',
 'ATSB_m_2005_215',
 'ATSB_a_2021_005',
 'ATSB_m_2008_012',
 'ATSB_r_2015_007',
 'ATSB_m_2001_163',
 'ATSB_a_2002_710',
 'ATSB_r_2014_024',
 'ATSB_m_2006_234',
 'ATSB_r_2004_004',
 'ATSB_a_2003_980',
 'ATSB_a_2017_105',
 'ATSB_a_2014_096',
 'ATSB_m_2013_011']

recommendations = pd.read_pickle(os.path.join(output_folder, "extracted_reports.pkl"))
recommendations.set_index("report_id", inplace=True)
recommendations = recommendations[["text", "headers", "toc", "recommendations", "important_text_recommendation"]]
recommendations = recommendations.loc[needed_ids]

# Normalize recommendations into list-of-dicts with required keys
rec_fields = ["recommendation", "recommendation_id", "recipient", "recommendation_context", "made"]

def normalize_recommendations(value):
    if value is None:
        return None
    # DataFrame path (most common in this dataset)
    if isinstance(value, pd.DataFrame):
        if value.empty:
            return None
        df = value.copy()
        for col in rec_fields:
            if col not in df.columns:
                raise ValueError(f"Missing expected recommendation field: {col}")
        return df[rec_fields].to_dict(orient="records")
    return None

recommendations_for_tests = (
    recommendations
    .rename(columns={"important_text_recommendation": "recommendation_section"})
    .assign(recommendations=lambda df: df["recommendations"].apply(normalize_recommendations))
)[["text", "headers", "toc", "recommendation_section", "recommendations"]]

recommendations_for_tests.to_pickle(os.path.join(test_data_folder, "recommendation_test_data.pkl"))